In [2]:
import pandas as pd

customers = pd.read_csv('../data/olist_customers_dataset.csv')
orders = pd.read_csv('../data/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

df = orders.merge(customers, on='customer_id')
df = df.merge(order_items, on='order_id')

df = df[df['order_status'] == 'delivered'].copy()

df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

paying_users = (
    df.groupby('order_month')['customer_unique_id']
    .nunique()
    .reset_index()
    .rename(columns={'customer_unique_id': 'paying_users'})
)

first_order = (
    df.groupby('customer_unique_id')['order_month']
    .min()
    .reset_index()
    .rename(columns={'order_month': 'first_month'})
)

df = df.merge(first_order, on='customer_unique_id')

df['cohort_index'] = (
    (df['order_month'].dt.year - df['first_month'].dt.year) * 12 +
    (df['order_month'].dt.month - df['first_month'].dt.month)
)

cohort = (
    df.groupby(['first_month', 'cohort_index'])['customer_unique_id']
    .nunique()
    .reset_index()
)

cohort_pivot = cohort.pivot(
    index='first_month',
    columns='cohort_index',
    values='customer_unique_id'
)

cohort_size = cohort_pivot[0]

retention = cohort_pivot.divide(cohort_size, axis=0)

retention_1 = retention[1].dropna().reset_index()
retention_1.columns = ['cohort', 'retention']

june_users = paying_users[
    paying_users['order_month'] == pd.Period('2017-06')
]['paying_users'].values[0]

retention_median = retention_1['retention'].median()

if retention_median < 0.005:
    retention_median = 0.01

expected_orders = june_users * retention_median

def get_impact(x):
    if x <= 50: return 1
    elif x <= 150: return 2
    elif x <= 350: return 3
    elif x <= 750: return 4
    elif x <= 1550: return 5
    elif x <= 3150: return 6
    elif x <= 6350: return 7
    elif x <= 12750: return 8
    elif x <= 25550: return 9
    else: return 10

base_impact = get_impact(expected_orders)

ice = pd.DataFrame({
    'Гипотеза': [
        'Исправить отмены заказов',
        'Сократить время доставки',
        'Добавить новый способ оплаты'
    ],
    'Impact': [
        min(base_impact + 2, 10),
        base_impact,
        min(base_impact + 1, 10)
    ],
    'Confidence': [8, 10, 9],
    'Ease': [6, 4, 5]
})

ice['ICE'] = (ice['Impact'] * ice['Confidence'] * ice['Ease']) / 3

ice = ice.sort_values('ICE', ascending=False)

ice

,Гипотеза,Impact,Confidence,Ease,ICE
0,Исправить отмены заказов,3,8,6,48.000000
2,Добавить новый способ оплаты,2,9,5,30.000000
1,Сократить время доставки,1,10,4,13.333333


Для оценки продуктовых гипотез был использован фреймворк ICE (Impact, Confidence, Ease).

Показатель Impact был рассчитан на основе ожидаемого количества дополнительных заказов. Для этого были использованы данные за июнь 2017 года и медианное значение retention первого месяца, которое интерпретируется как вероятность совершения повторного заказа. Ожидаемое количество дополнительных заказов было получено как произведение числа платящих пользователей на уровень retention, после чего значение было переведено в шкалу Impact.

Следует учитывать, что данный способ расчета Impact имеет ограничения. Он отражает влияние гипотез преимущественно на повторные заказы и может занижать эффект тех изменений, которые затрагивают все заказы. В частности, гипотеза об устранении отмен заказов влияет на общий объем доставленных заказов, а не только на повторные покупки, поэтому её реальный эффект может быть выше рассчитанного.

По результатам оценки гипотез были получены следующие значения ICE:

- Исправление отмен заказов — ICE = 48
- Добавление нового способа оплаты — ICE = 30
- Сокращение времени доставки — ICE = 13.3

Наибольшее значение ICE получила гипотеза исправления багов в системе процессинга заказов.

Данная гипотеза оказывает прямое влияние на количество успешно завершённых заказов, снижает потери на этапе оформления и повышает общий объем продаж (GMV). В отличие от других гипотез, её эффект проявляется на всей пользовательской базе, а не только на сегменте пользователей, совершающих повторные покупки.

Таким образом, приоритетной для реализации является гипотеза по устранению отмен заказов, так как она обеспечивает максимальный вклад в рост ключевых бизнес-метрик маркетплейса.